### Nota sobre el modo `overwrite`
El enunciado pide escribir en modo `overwrite`. Al escribir por JDBC en este modo, Spark **elimina y vuelve a crear** cada tabla de destino con los tipos del DataFrame. Por eso las restricciones del DDL (PRIMARY KEY, DEFAULT e índices) no se conservan. Para compensarlo, el script garantiza en el propio DataFrame la unicidad de las llaves y que no haya nulos en las columnas NOT NULL.

In [1]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.window import Window

# Parcial 2 Procesamiento de grades volumenes de datos
# Integrantes: Juan Diego Valencia y Federico Teran

# Inicializacion de la sesion de spark
spark = (SparkSession.builder
    .appName("ETL-DataMart-PlayerPerformance")
    .master("local[*]")
    .config("spark.driver.memory", "1g")
    .config("spark.executor.memory", "1500m")
    .config("spark.sql.extensions", "io.delta.sql.DeltaSparkSessionExtension")
    .config("spark.sql.catalog.spark_catalog", "org.apache.spark.sql.delta.catalog.DeltaCatalog")
    .config("spark.hadoop.fs.s3a.endpoint", "http://localhost:9000")
    .config("spark.hadoop.fs.s3a.access.key", "minioadmin")
    .config("spark.hadoop.fs.s3a.secret.key", "minioadmin123")
    .config("spark.hadoop.fs.s3a.path.style.access", "true")
    .config("spark.hadoop.fs.s3a.impl", "org.apache.hadoop.fs.s3a.S3AFileSystem")
    .config("spark.hadoop.fs.s3a.connection.ssl.enabled", "false")
    .config("spark.hadoop.fs.s3a.aws.credentials.provider", "org.apache.hadoop.fs.s3a.SimpleAWSCredentialsProvider")
    .getOrCreate())

jdbc_url = "jdbc:postgresql://localhost:5432/db_sports"
dw_url = "jdbc:postgresql://localhost:5432/dwh_sports"

properties = {"user": "postgres", "password": "123abc", "driver": "org.postgresql.Driver"}

print("SparkSession creada exitosamente")
print(f"App: {spark.sparkContext.appName}")
print(f"Version: {spark.version}")
print(f"BD operacional: {jdbc_url}")
print(f"Data Warehouse: {dw_url}")

26/10/06 16:49:47 WARN Utils: Your hostname, fede-Aspire-A315-57G resolves to a loopback address: 127.0.1.1; using 10.62.21.191 instead (on interface wlp4s0)
26/10/06 16:49:47 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/10/06 16:49:47 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


SparkSession creada exitosamente
App: ETL-DataMart-PlayerPerformance
Version: 3.5.8
BD operacional: jdbc:postgresql://localhost:5432/db_sports
Data Warehouse: jdbc:postgresql://localhost:5432/dwh_sports


## (Opcional) Carga de los CSV al bucket `rawdata` del Data Lake
El enunciado indica que `person_contacts.csv` y `person_memberships.csv` ya están en el bucket `rawdata` de MinIO. Esta celda solo es necesaria si el bucket o los archivos no existen todavía. Sigue el mismo patrón de la **Guía 02-B (Celda 3)**: un cliente `boto3` apuntando a MinIO, más `head_bucket` y `create_bucket`.

Los archivos locales están en la carpeta `data/`, junto a este notebook.

In [2]:
import boto3
from botocore.exceptions import ClientError

print("=" * 60)
print("CARGA DE ARCHIVOS CSV AL DATA LAKE (MINIO)")
print("=" * 60)

# Cliente S3 apuntando a MinIO
s3 = boto3.client(
    's3',
    endpoint_url='http://localhost:9000',
    aws_access_key_id='minioadmin',
    aws_secret_access_key='minioadmin123',
    region_name='us-east-1'
)

# Crear bucket si no existe
bucket_name = "rawdata"
try:
    s3.head_bucket(Bucket=bucket_name)
    print(f"Bucket '{bucket_name}' ya existe")
except ClientError:
    s3.create_bucket(Bucket=bucket_name)
    print(f"Bucket '{bucket_name}' creado")

# Subir los archivos CSV (archivo local -> bucket rawdata)
for file_name in ["person_contacts.csv", "person_memberships.csv"]:
    s3.upload_file(f"data/{file_name}", bucket_name, file_name)
    print(f"Archivo '{file_name}' cargado en s3a://{bucket_name}/{file_name}")

CARGA DE ARCHIVOS CSV AL DATA LAKE (MINIO)
Bucket 'rawdata' ya existe


Archivo 'person_contacts.csv' cargado en s3a://rawdata/person_contacts.csv
Archivo 'person_memberships.csv' cargado en s3a://rawdata/person_memberships.csv


## 1. Carga y transformación de dimensiones secundarias (15%)

### 1.1 Dimensión de Eventos (`dim_events`)
Se sigue el patrón de `dim_bodyparts` de la **Guía 03**: `select` + `distinct`, una ventana `Window.orderBy` y `F.row_number()` como llave subrogada.

La columna operacional `id` se renombra a `entity_event_id` para conservar la trazabilidad con el origen y para ordenar la ventana. El DDL de `dim_events` no tiene esa columna, así que se descarta en el `select` final.

In [3]:
# ============================================================
# 1.1 Dimensión de Eventos (dim_events)
# ============================================================

# Extraer la tabla operacional de eventos
df_events = spark.read.jdbc(url=jdbc_url, table="events", properties=properties)

df_events.printSchema()

# Renombrar 'id' -> 'entity_event_id' y seleccionar de forma única los campos del evento
df_events_unique = df_events.withColumnRenamed("id", "entity_event_id") \
    .select(
        F.col("entity_event_id"),
        F.col("event_key"),
        F.col("event_status"),
        F.col("start_date_time").alias("start_date")
    ).distinct()

# Generar llave subrogada incremental usando row_number()
window_event = Window.orderBy("entity_event_id")
df_dim_events = df_events_unique.withColumn("id", F.row_number().over(window_event)) \
    .select(
        F.col("id").cast("integer"),
        F.col("event_key").cast("string"),
        F.col("event_status").cast("string"),
        F.col("start_date").cast("timestamp")
    )

df_dim_events.show(10, truncate=False)

# Guardar en PostgreSQL (Data Warehouse)
df_dim_events.write.jdbc(url=dw_url, table="dim_events", mode="overwrite", properties=properties)

print(f"Dimensión dim_events cargada exitosamente. Registros: {df_dim_events.count()}")

root
 |-- id: integer (nullable = true)
 |-- event_key: string (nullable = true)
 |-- publisher_id: integer (nullable = true)
 |-- start_date_time: timestamp (nullable = true)
 |-- site_id: integer (nullable = true)
 |-- site_alignment: string (nullable = true)
 |-- event_status: string (nullable = true)
 |-- duration: string (nullable = true)
 |-- attendance: string (nullable = true)
 |-- last_update: timestamp (nullable = true)



26/10/06 16:31:28 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/10/06 16:31:28 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/10/06 16:31:28 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.


26/10/06 16:31:30 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/10/06 16:31:30 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/10/06 16:31:30 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.


+---+----------------------+------------+-------------------+
|id |event_key             |event_status|start_date         |
+---+----------------------+------------+-------------------+
|1  |l.mlb.com-2007-e.19860|post-event  |2007-08-04 00:10:00|
|2  |l.mlb.com-2007-e.20721|post-event  |2007-08-04 02:40:00|
|3  |l.mlb.com-2007-e.19873|post-event  |2007-04-02 20:05:00|
|4  |l.mlb.com-2007-e.19874|post-event  |2007-04-04 00:35:00|
|5  |l.mlb.com-2007-e.19876|post-event  |2007-04-04 19:05:00|
|6  |l.mlb.com-2007-e.19516|post-event  |2007-04-05 23:05:00|
|7  |l.mlb.com-2007-e.19517|post-event  |2007-04-06 23:05:00|
|8  |l.mlb.com-2007-e.19519|post-event  |2007-04-07 23:05:00|
|9  |l.mlb.com-2007-e.19520|post-event  |2007-04-08 17:35:00|
|10 |l.mlb.com-2007-e.18783|post-event  |2007-04-10 01:40:00|
+---+----------------------+------------+-------------------+
only showing top 10 rows



26/10/06 16:31:31 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/10/06 16:31:31 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/10/06 16:31:31 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.


26/10/06 16:31:31 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/10/06 16:31:31 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/10/06 16:31:32 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/10/06 16:31:32 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.


Dimensión dim_events cargada exitosamente. Registros: 4350


### 1.2 Dimensión de Posiciones (`dim_positions`)
Es el mismo patrón de `dim_bodyparts` de la **Guía 03**: abreviaturas únicas sin nulos y una llave subrogada ordenada por la abreviatura.

In [4]:
# ============================================================
# 1.2 Dimensión de Posiciones (dim_positions)
# ============================================================

# Extraer la tabla operacional de posiciones
df_positions = spark.read.jdbc(url=jdbc_url, table="positions", properties=properties)

df_positions.show(10)

# Obtener abreviaturas únicas descartando nulos
df_abbreviations = df_positions.select("abbreviation") \
    .filter(F.col("abbreviation").isNotNull()) \
    .distinct()

# Generar llave subrogada incremental usando row_number()
window_position = Window.orderBy("abbreviation")
df_dim_positions = df_abbreviations.withColumn("id", F.row_number().over(window_position)) \
    .select(
        F.col("id").cast("integer"),
        F.col("abbreviation").cast("string")
    )

df_dim_positions.show(10)

# Guardar en PostgreSQL (Data Warehouse)
df_dim_positions.write.jdbc(url=dw_url, table="dim_positions", mode="overwrite", properties=properties)

print(f"Dimensión dim_positions cargada exitosamente. Registros: {df_dim_positions.count()}")

+---+--------------+------------+
| id|affiliation_id|abbreviation|
+---+--------------+------------+
|  1|             2|           8|
|  2|             2|           5|
|  3|             2|           2|
|  4|             2|           3|
|  5|             2|          dh|
|  6|             2|           6|
|  7|             2|           7|
|  8|             2|           9|
|  9|             2|           4|
| 10|             2|           1|
+---+--------------+------------+
only showing top 10 rows



26/10/06 16:31:33 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/10/06 16:31:33 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/10/06 16:31:33 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/10/06 16:31:33 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/10/06 16:31:33 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/10/06 16:31:33 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.


+---+------------+
| id|abbreviation|
+---+------------+
|  1|           1|
|  2|          1B|
|  3|       1B-3B|
|  4|       1B-RF|
|  5|           2|
|  6|          2B|
|  7|           3|
|  8|          3B|
|  9|       3B-SS|
| 10|           4|
+---+------------+
only showing top 10 rows



26/10/06 16:31:33 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/10/06 16:31:33 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/10/06 16:31:33 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/10/06 16:31:33 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/10/06 16:31:33 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/10/06 16:31:33 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/10/06 1

Dimensión dim_positions cargada exitosamente. Registros: 70


## 2. ETL multifuente para `dim_persons_performance` (20%)

Se parte del patrón de `dim_persons` de la **Guía 03** (LEFT JOIN `persons` con `display_names`, `coalesce` con `"N/A"`, `row_number`, `version`, `date_from` y `date_to`) y se enriquece con los dos CSV del Data Lake.

**Paso 2.1: Datos operacionales.** El enunciado pide cruzar `persons.id` con `display_names.entity_id`. En `display_names` el campo `entity_id` se repite entre tipos de entidad (`persons`, `teams`, `affiliations`). Por eso primero se filtra `entity_type == "persons"`; de lo contrario, una misma persona quedaría duplicada en la dimensión.

In [5]:
# ============================================================
# 2.1 Extracción de la BD operacional: persons y display_names
# ============================================================
df_persons = spark.read.jdbc(url=jdbc_url, table="persons", properties=properties)
df_display_names = spark.read.jdbc(url=jdbc_url, table="display_names", properties=properties)

# Solo los nombres de entidades de tipo persona (entity_id se repite entre persons/teams/affiliations)
df_display_names_persons = df_display_names.filter(F.col("entity_type") == "persons")

# LEFT JOIN persons -> display_names, coalesce en nombres y estandarización del género
df_persons_clean = df_persons.join(df_display_names_persons, df_persons["id"] == df_display_names_persons["entity_id"], "left") \
    .select(
        df_persons["id"].alias("entity_id"),
        F.coalesce(F.col("first_name"), F.lit("N/A")).alias("first_name"),
        F.coalesce(F.col("last_name"), F.lit("N/A")).alias("last_name"),
        F.lower(F.trim(F.col("gender"))).alias("gender")
    ).distinct()

df_persons_clean.show(10)

+---------+----------+---------+------+
|entity_id|first_name|last_name|gender|
+---------+----------+---------+------+
|      148|     Derek|     Lowe|  NULL|
|      463|     Danny|   Putnam|  NULL|
|      471|      Pete| LaForest|  NULL|
|      496|     Manny|Delcarmen|  NULL|
|      833|     Boone|    Logan|  NULL|
|     1088|     Corky|   Miller|  NULL|
|     1238|      Dre'|      Bly|  NULL|
|     1342|    Jamaal| Anderson|  NULL|
|     1580|     Teddy|   Lehman|  NULL|
|     1591|       Dan| Orlovsky|  NULL|
+---------+----------+---------+------+
only showing top 10 rows



**Paso 2.2: Ingesta desde el Data Lake (S3A).** Se leen los CSV de la capa raw (`s3a://rawdata/`), con la cabecera activada y la inferencia de esquema.

> **Inferencia mínima:** la Guía 02 lee desde MinIO con `spark.read.format("delta").load("s3a://...")`. Aquí se usa la misma forma, pero con `format("csv")` y las opciones `header` e `inferSchema` que exige el enunciado.

**Reglas de limpieza:** los correos van en minúsculas y sin espacios (`F.lower(F.trim(...))`), y los niveles de membresía con la inicial en mayúscula (`F.initcap(F.trim(...))`).

In [6]:
# ============================================================
# 2.2 Extracción del Data Lake: archivos CSV en s3a://rawdata/
# ============================================================
contacts_path = "s3a://rawdata/person_contacts.csv"
memberships_path = "s3a://rawdata/person_memberships.csv"

df_contacts = spark.read.format("csv") \
    .option("header", "true") \
    .option("inferSchema", "true") \
    .load(contacts_path)

df_memberships = spark.read.format("csv") \
    .option("header", "true") \
    .option("inferSchema", "true") \
    .load(memberships_path)

df_contacts.printSchema()
df_memberships.printSchema()

# Limpieza del archivo de contactos: correos en minúscula y sin espacios
df_contacts_clean = df_contacts.select(
    F.col("person_id").cast("integer"),
    F.lower(F.trim(F.col("email"))).alias("email")
)

# Limpieza del archivo de membresías: niveles con formato capitalizado
df_memberships_clean = df_memberships.select(
    F.col("person_id").cast("integer"),
    F.initcap(F.trim(F.col("membership_level"))).alias("membership_level")
)

df_contacts_clean.show(5, truncate=False)
df_memberships_clean.show(5)

26/10/06 16:31:35 WARN MetricsConfig: Cannot locate configuration: tried hadoop-metrics2-s3a-file-system.properties,hadoop-metrics2.properties


root
 |-- person_id: integer (nullable = true)
 |-- email: string (nullable = true)

root
 |-- person_id: integer (nullable = true)
 |-- membership_level: string (nullable = true)

+---------+---------------------------+
|person_id|email                      |
+---------+---------------------------+
|1        |john.doe@example.com       |
|2        |sarah.smith@sportmail.org  |
|3        |michael.jones@athlete.net  |
|4        |emily.brown@sportsmail.com |
|5        |david.wilson@performance.io|
+---------+---------------------------+
only showing top 5 rows



+---------+----------------+
|person_id|membership_level|
+---------+----------------+
|        1|            Gold|
|        2|          Silver|
|        3|        Platinum|
|        4|        Standard|
|        5|            Gold|
+---------+----------------+
only showing top 5 rows



**Paso 2.3: Integración (LEFT JOINs sucesivos) y metadatos de control SCD.** Las personas que no aparecen en los CSV reciben los valores por defecto `"no-email@domain.com"` y `"Standard"`.

In [7]:
# ============================================================
# 2.3 Integración: LEFT JOIN con contactos y membresías
# ============================================================
df_persons_enriched = df_persons_clean \
    .join(df_contacts_clean, df_persons_clean["entity_id"] == df_contacts_clean["person_id"], "left") \
    .join(df_memberships_clean, df_persons_clean["entity_id"] == df_memberships_clean["person_id"], "left") \
    .select(
        df_persons_clean["entity_id"],
        df_persons_clean["first_name"],
        df_persons_clean["last_name"],
        df_persons_clean["gender"],
        F.coalesce(F.col("email"), F.lit("no-email@domain.com")).alias("email"),
        F.coalesce(F.col("membership_level"), F.lit("Standard")).alias("membership_level")
    )

# Generar llave subrogada (id) y atributos de control SCD
window_person = Window.orderBy("entity_id")
df_dim_persons_performance = df_persons_enriched.withColumn("id", F.row_number().over(window_person)) \
    .select(
        F.col("id").cast("integer"),
        F.col("entity_id").cast("integer"),
        F.col("first_name").cast("string"),
        F.col("last_name").cast("string"),
        F.col("gender").cast("string"),
        F.col("email").cast("string"),
        F.col("membership_level").cast("string"),
        F.lit(1).cast("integer").alias("version"),
        F.current_timestamp().alias("date_from"),
        F.lit(None).cast("timestamp").alias("date_to")
    )

df_dim_persons_performance.show(10, truncate=False)

# Guardar en PostgreSQL (Data Warehouse)
df_dim_persons_performance.write.jdbc(url=dw_url, table="dim_persons_performance", mode="overwrite", properties=properties)

print(f"Dimensión dim_persons_performance cargada exitosamente. Registros: {df_dim_persons_performance.count()}")

26/10/06 16:31:37 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/10/06 16:31:37 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/10/06 16:31:37 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/10/06 16:31:37 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/10/06 16:31:37 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/10/06 16:31:37 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/10/06 1

26/10/06 16:31:37 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/10/06 16:31:37 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/10/06 16:31:37 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/10/06 16:31:37 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/10/06 16:31:37 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.


+---+---------+----------+---------+------+---------------------------+----------------+-------+--------------------------+-------+
|id |entity_id|first_name|last_name|gender|email                      |membership_level|version|date_from                 |date_to|
+---+---------+----------+---------+------+---------------------------+----------------+-------+--------------------------+-------+
|1  |1        |Grady     |Sizemore |NULL  |john.doe@example.com       |Gold            |1      |2026-10-06 16:31:37.138929|NULL   |
|2  |2        |Casey     |Blake    |NULL  |sarah.smith@sportmail.org  |Silver          |1      |2026-10-06 16:31:37.138929|NULL   |
|3  |3        |Victor    |Martinez |NULL  |michael.jones@athlete.net  |Platinum        |1      |2026-10-06 16:31:37.138929|NULL   |
|4  |4        |Ryan      |Garko    |NULL  |emily.brown@sportsmail.com |Standard        |1      |2026-10-06 16:31:37.138929|NULL   |
|5  |5        |Travis    |Hafner   |NULL  |david.wilson@performance.io|Gold 

26/10/06 16:31:37 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/10/06 16:31:37 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/10/06 16:31:37 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/10/06 16:31:37 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/10/06 16:31:37 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/10/06 16:31:38 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/10/06 1

26/10/06 16:31:38 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/10/06 16:31:38 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/10/06 16:31:38 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/10/06 16:31:38 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/10/06 16:31:38 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.


Dimensión dim_persons_performance cargada exitosamente. Registros: 3937


## 3. Carga de la tabla de hechos `ft_player_performance` (35%)

Se sigue el patrón de `ft_injuries` de la **Guía 03**: se leen las dimensiones ya guardadas en el DW para recuperar las llaves sustitutas, se resuelven las llaves foráneas con joins y se construye `dateid` con `F.date_format(..., "yyyyMMdd").cast("integer")`.

**Joins utilizados:**
1. `participants_events` **LEFT JOIN** `positions` (`participant_id == affiliation_id`): asocia el rol sin perder participaciones, porque la mayoría no tiene posición registrada.
2. **INNER JOIN** con `events` (`event_id == id`) para obtener la clave de negocio `event_key` y la fecha de inicio.
3. **INNER JOIN** con `dim_events` (por `event_key`) y con `dim_persons_performance` (`participant_id == entity_id`).
4. **LEFT JOIN** con `dim_positions` (por `abbreviation`). Como `id_position` es NOT NULL, las participaciones sin posición quedan con `id_position = 0`.

In [8]:
# ============================================================
# 3.1 Lectura de las dimensiones ya almacenadas en el DW
# ============================================================
dw_dim_events = spark.read.jdbc(url=dw_url, table="dim_events", properties=properties)
dw_dim_persons = spark.read.jdbc(url=dw_url, table="dim_persons_performance", properties=properties)
dw_dim_positions = spark.read.jdbc(url=dw_url, table="dim_positions", properties=properties)

# ============================================================
# 3.2 Extracción operacional
# ============================================================
df_participants_events = spark.read.jdbc(url=jdbc_url, table="participants_events", properties=properties)
df_positions_src = spark.read.jdbc(url=jdbc_url, table="positions", properties=properties)
df_events_src = spark.read.jdbc(url=jdbc_url, table="events", properties=properties)

df_participants_events.show(5)

+---+----------------+--------------+--------+---------+-----+-------------+----+
| id|participant_type|participant_id|event_id|alignment|score|event_outcome|rank|
+---+----------------+--------------+--------+---------+-----+-------------+----+
| 77|           teams|            26|      39|     away|    3|          win|NULL|
| 78|           teams|            17|      39|     home|    1|         loss|NULL|
| 79|           teams|            26|      40|     away|    4|         loss|NULL|
| 80|           teams|            17|      40|     home|   10|          win|NULL|
| 81|           teams|            26|      41|     away|    2|         loss|NULL|
+---+----------------+--------------+--------+---------+-----+-------------+----+
only showing top 5 rows



In [9]:
# ============================================================
# 3.3 Resolución de integridad referencial y joins
# ============================================================

# Participación -> posición (por identificador de afiliación/participante)
# Participación -> evento operacional (para alinear la clave de negocio event_key)
df_participation = df_participants_events \
    .join(df_positions_src, df_participants_events["participant_id"] == df_positions_src["affiliation_id"], "left") \
    .join(df_events_src, df_participants_events["event_id"] == df_events_src["id"], "inner") \
    .select(
        df_participants_events["participant_id"],
        df_participants_events["score"],
        df_participants_events["rank"],
        df_positions_src["abbreviation"],
        df_events_src["event_key"],
        df_events_src["start_date_time"]
    )

# Resolver llaves sustitutas haciendo join con las dimensiones del DW
df_facts = df_participation \
    .join(dw_dim_events, df_participation["event_key"] == dw_dim_events["event_key"], "inner") \
    .join(dw_dim_persons, df_participation["participant_id"] == dw_dim_persons["entity_id"], "inner") \
    .join(dw_dim_positions, df_participation["abbreviation"] == dw_dim_positions["abbreviation"], "left")

**Transformación y cálculo de métricas:**
- `dateid`: fecha de inicio del evento en formato `YYYYMMDD`. Si la fecha es nula, se usa la fecha actual.
- `score_numeric`: el score (texto en el origen) se castea a numérico; si es nulo, queda en `0.00`.
- `rank`: los nulos se inicializan en `0`.
- Desduplicación estricta sobre la llave primaria compuesta `(dateid, id_event, id_person)`.

> **Inferencia mínima:** la Guía 03 solo usa `.distinct()`, que elimina filas idénticas en *todas* las columnas. Aquí una participación puede cruzar con varias posiciones, así que dos filas pueden compartir la llave primaria y diferir en `id_position`. Para desduplicar *sobre la clave primaria* se usa `.dropDuplicates([...])`, que es el equivalente de `distinct()` restringido a un subconjunto de columnas.

In [10]:
# ============================================================
# 3.4 Transformación y cálculo de métricas
# ============================================================
df_ft_player_performance = df_facts.select(
    F.date_format(F.coalesce(df_participation["start_date_time"], F.current_timestamp()), "yyyyMMdd").cast("integer").alias("dateid"),
    dw_dim_events["id"].cast("integer").alias("id_event"),               # id de dim_events
    dw_dim_persons["id"].cast("integer").alias("id_person"),             # id de dim_persons_performance
    F.coalesce(dw_dim_positions["id"], F.lit(0)).cast("integer").alias("id_position"),  # id de dim_positions (0 = sin posición)
    F.coalesce(df_participation["score"].cast("decimal(10,2)"), F.lit(0.00)).cast("decimal(10,2)").alias("score_numeric"),
    F.coalesce(df_participation["rank"], F.lit(0)).cast("integer").alias("rank")
)

# Desduplicación estricta sobre la clave primaria compuesta
df_ft_player_performance = df_ft_player_performance.dropDuplicates(["dateid", "id_event", "id_person"])

df_ft_player_performance.show(10)

# ============================================================
# 3.5 Carga final en el Data Warehouse
# ============================================================
df_ft_player_performance.write.jdbc(url=dw_url, table="ft_player_performance", mode="overwrite", properties=properties)

print(f"Tabla de Hechos ft_player_performance cargada exitosamente. Registros: {df_ft_player_performance.count()}")

+--------+--------+---------+-----------+-------------+----+
|  dateid|id_event|id_person|id_position|score_numeric|rank|
+--------+--------+---------+-----------+-------------+----+
|20080330|    3173|      115|          0|         0.00|   0|
|20071201|    3914|      101|          0|         0.00|   0|
|20080117|    3547|       81|          0|         0.00|   0|
|20070621|     230|       28|          0|         8.00|   0|
|20070829|     284|       28|          0|         0.00|   0|
|20071206|    2638|      122|          0|         0.00|   0|
|20080120|    2912|       93|          0|         0.00|   0|
|20070516|    1423|        1|          0|         0.00|   0|
|20070722|    1049|       13|          0|         0.00|   0|
|20070423|     714|        6|          0|         5.00|   0|
+--------+--------+---------+-----------+-------------+----+
only showing top 10 rows



Tabla de Hechos ft_player_performance cargada exitosamente. Registros: 8700


### Verificación de la carga
Se releen las tablas desde PostgreSQL y se comparan los conteos, igual que la verificación de integridad de la **Guía 02-C**.

In [11]:
print("=" * 60)
print("VERIFICACIÓN DE LA CARGA EN EL DATA WAREHOUSE")
print("=" * 60)

for table_name in ["dim_events", "dim_positions", "dim_persons_performance", "ft_player_performance"]:
    df_verificacion = spark.read.jdbc(url=dw_url, table=table_name, properties=properties)
    print(f"{table_name}: {df_verificacion.count()} registros")

# Conteos cruzados de la tabla de hechos: total vs. llaves primarias únicas
df_ft_verificacion = spark.read.jdbc(url=dw_url, table="ft_player_performance", properties=properties)
count_total = df_ft_verificacion.count()
count_pk = df_ft_verificacion.select("dateid", "id_event", "id_person").distinct().count()

print(f"\nRegistros en ft_player_performance: {count_total}")
print(f"Llaves primarias únicas: {count_pk}")
print(f"Integridad verificada: {'CORRECTA' if count_total == count_pk else 'ERROR'}")

VERIFICACIÓN DE LA CARGA EN EL DATA WAREHOUSE
dim_events: 4350 registros


dim_positions: 70 registros
dim_persons_performance: 3937 registros


ft_player_performance: 8700 registros



Registros en ft_player_performance: 8700
Llaves primarias únicas: 8700
Integridad verificada: CORRECTA


## 4. Catálogo de consultas analíticas
Las consultas se ejecutan en PostgreSQL sobre la base `dwh_sports`, igual que en la **Guía 03** (`sudo -u postgres psql` y luego `\c dwh_sports`). El script completo está en `sql/consultas_analiticas.sql`.

**Consulta 1 (3%):** Top 10 de atletas con mayor puntuación promedio. Se cruza la tabla de hechos con `dim_persons_performance`, se cuentan los eventos y se promedia `score_numeric`, considerando solo atletas con nivel de membresía registrado.

```sql
-- (solo atletas con nivel de membresía registrado)
SELECT
    p.first_name,
    p.last_name,
    p.membership_level,
    COUNT(f.id_event) AS total_events,
    ROUND(AVG(f.score_numeric)::numeric, 2) AS avg_score
FROM ft_player_performance f
JOIN dim_persons_performance p ON f.id_person = p.id
WHERE p.membership_level IS NOT NULL
GROUP BY p.first_name, p.last_name, p.membership_level
ORDER BY avg_score DESC
LIMIT 10;
```

Resultado consulta:

```
 first_name | last_name | membership_level | total_events | avg_score 
------------+-----------+------------------+--------------+-----------
 Ben        | Francisco | Standard         |           16 |     22.19
 Jake       | Westbrook | Platinum         |            7 |     16.29
 Mike       | Rouse     | Gold             |           12 |     15.33
 Jensen     | Lewis     | Platinum         |           16 |     15.25
 Andy       | Marte     | Standard         |           16 |     14.19
 Tom        | Mastny    | Silver           |            8 |     13.63
 Paul       | Byrd      | Silver           |           16 |     12.69
 Orlando    | Hudson    | Standard         |           16 |     12.50
 David      | Dellucci  | Platinum         |           16 |     12.38
 Edward     | Mujica    | Gold             |            8 |     12.38
(10 rows)
```

**Consulta 2 (5%):** Rendimiento segmentado por nivel de membresía y género: atletas únicos, participaciones totales, puntaje promedio y puntaje máximo por segmento.

```sql
SELECT
    p.membership_level,
    p.gender,
    COUNT(DISTINCT f.id_person) AS unique_athletes,
    COUNT(f.id_event) AS total_participations,
    ROUND(AVG(f.score_numeric)::numeric, 2) AS avg_score,
    ROUND(MAX(f.score_numeric)::numeric, 2) AS max_score
FROM ft_player_performance f
JOIN dim_persons_performance p ON f.id_person = p.id
GROUP BY p.membership_level, p.gender
ORDER BY p.membership_level, avg_score DESC;
```

Resultado consulta:

```
 membership_level | gender | unique_athletes | total_participations | avg_score | max_score 
------------------+--------+-----------------+----------------------+-----------+-----------
 Gold             |        |              14 |                 1389 |      3.55 |     41.00
 Platinum         |        |              13 |                 1373 |      3.58 |     45.00
 Silver           |        |              10 |                 1042 |      3.60 |     56.00
 Standard         |        |              85 |                 4896 |      2.39 |    120.00
(4 rows)
```

**Consulta 3 (5%):** Impacto del estado del evento sobre las posiciones de juego. Cruza los hechos con `dim_positions` y `dim_events` y promedia el puntaje por posición y estado del evento. Las participaciones sin posición (`id_position = 0`) no tienen pareja en `dim_positions`, así que el `JOIN` las excluye.

```sql
SELECT
    pos.abbreviation AS position_code,
    e.event_status,
    COUNT(f.id_event) AS total_events,
    ROUND(AVG(f.score_numeric)::numeric, 2) AS avg_position_score
FROM ft_player_performance f
JOIN dim_positions pos ON f.id_position = pos.id
JOIN dim_events e ON f.id_event = e.id
GROUP BY pos.abbreviation, e.event_status
HAVING COUNT(f.id_event) > 0
ORDER BY avg_position_score DESC;
```

Resultado consulta:

```
 position_code  | event_status | total_events | avg_position_score 
----------------+--------------+--------------+--------------------
 RF-3B          | post-event   |          110 |               5.18
 forward,center | post-event   |          111 |               4.68
 safety-free    | post-event   |          109 |               4.65
 goalie         | post-event   |          111 |               4.11
 safety-free    | pre-event    |           53 |               0.00
 forward,center | pre-event    |           51 |               0.00
 RF-3B          | postponed    |            7 |               0.00
 goalie         | pre-event    |           51 |               0.00
 safety-free    | postponed    |            2 |               0.00
 RF-3B          | pre-event    |           52 |               0.00
(10 rows)
```

**Consulta 4 (7%):** Top 3 de atletas por categoría de membresía usando `DENSE_RANK()` particionado por `membership_level`.

```sql
WITH athlete_scores AS (
    SELECT
        p.id,
        p.first_name || ' ' || p.last_name AS full_name,
        p.membership_level,
        AVG(f.score_numeric) AS avg_score,
        DENSE_RANK() OVER (PARTITION BY p.membership_level ORDER BY AVG(f.score_numeric) DESC) AS ranking
    FROM ft_player_performance f
    JOIN dim_persons_performance p ON f.id_person = p.id
    GROUP BY p.id, p.first_name || ' ' || p.last_name, p.membership_level
)
SELECT
    membership_level,
    full_name,
    ROUND(avg_score::numeric, 2) AS avg_score,
    ranking
FROM athlete_scores
WHERE ranking <= 3
ORDER BY membership_level, ranking;
```

Resultado consulta:

```
 membership_level |   full_name    | avg_score | ranking 
------------------+----------------+-----------+---------
 Gold             | Mike Rouse     |     15.33 |       1
 Gold             | Edward Mujica  |     12.38 |       2
 Gold             | Jason Stanford |     11.81 |       3
 Platinum         | Jake Westbrook |     16.29 |       1
 Platinum         | Jensen Lewis   |     15.25 |       2
 Platinum         | David Dellucci |     12.38 |       3
 Silver           | Tom Mastny     |     13.63 |       1
 Silver           | Paul Byrd      |     12.69 |       2
 Silver           | Jason Michaels |      9.11 |       3
 Standard         | Ben Francisco  |     22.19 |       1
 Standard         | Andy Marte     |     14.19 |       2
 Standard         | Orlando Hudson |     12.50 |       3
(12 rows)
```

**Consulta 5 (10%):** Rendimiento individual frente al promedio de su categoría: desviación respecto al promedio del tier, clasificación con `CASE` y percentil dentro de la categoría con `PERCENT_RANK()`.

```sql
WITH tier_stats AS (
    SELECT
        p.membership_level,
        AVG(f.score_numeric) AS tier_avg_score
    FROM ft_player_performance f
    JOIN dim_persons_performance p ON f.id_person = p.id
    GROUP BY p.membership_level
),
participant_performance AS (
    SELECT
        p.id,
        p.first_name || ' ' || p.last_name AS full_name,
        p.membership_level,
        COUNT(f.id_event) AS total_events,
        AVG(f.score_numeric) AS personal_avg_score
    FROM ft_player_performance f
    JOIN dim_persons_performance p ON f.id_person = p.id
    GROUP BY p.id, p.first_name || ' ' || p.last_name, p.membership_level
)
SELECT
    pp.full_name,
    pp.membership_level,
    ROUND(pp.personal_avg_score::numeric, 2) AS personal_avg_score,
    ROUND(ts.tier_avg_score::numeric, 2) AS tier_avg_score,
    ROUND((pp.personal_avg_score - ts.tier_avg_score)::numeric, 2) AS difference_vs_tier,
    CASE
        WHEN pp.personal_avg_score >= ts.tier_avg_score THEN 'Por encima del promedio'
        ELSE 'Por debajo del promedio'
    END AS performance_status,
    ROUND(PERCENT_RANK() OVER (PARTITION BY pp.membership_level ORDER BY pp.personal_avg_score)::numeric, 2) AS percentile_in_tier
FROM participant_performance pp
JOIN tier_stats ts ON pp.membership_level = ts.membership_level
ORDER BY pp.membership_level, pp.personal_avg_score DESC;
```

Resultado consulta:

```
     full_name      | membership_level | personal_avg_score | tier_avg_score | difference_vs_tier |   performance_status    | percentile_in_tier 
--------------------+------------------+--------------------+----------------+--------------------+-------------------------+--------------------
 Mike Rouse         | Gold             |              15.33 |           3.55 |              11.78 | Por encima del promedio |               1.00
 Edward Mujica      | Gold             |              12.38 |           3.55 |               8.82 | Por encima del promedio |               0.92
 Jason Stanford     | Gold             |              11.81 |           3.55 |               8.26 | Por encima del promedio |               0.85
 Rafael Perez       | Gold             |              10.31 |           3.55 |               6.76 | Por encima del promedio |               0.77
 Shin-Soo Choo      | Gold             |               8.13 |           3.55 |               4.57 | Por encima del promedio |               0.69
 Trot Nixon         | Gold             |               6.83 |           3.55 |               3.28 | Por encima del promedio |               0.62
 Jason Kubel        | Gold             |               3.65 |           3.55 |               0.09 | Por encima del promedio |               0.54
 Dennys Reyes       | Gold             |               3.30 |           3.55 |              -0.25 | Por debajo del promedio |               0.46
 Joe Mauer          | Gold             |               3.20 |           3.55 |              -0.35 | Por debajo del promedio |               0.38
 Josh Barfield      | Gold             |               3.10 |           3.55 |              -0.45 | Por debajo del promedio |               0.31
 Travis Hafner      | Gold             |               3.09 |           3.55 |              -0.47 | Por debajo del promedio |               0.23
 Rafael Betancourt  | Gold             |               2.93 |           3.55 |              -0.62 | Por debajo del promedio |               0.15
 Grady Sizemore     | Gold             |               2.87 |           3.55 |              -0.68 | Por debajo del promedio |               0.08
 Mike Redmond       | Gold             |               2.82 |           3.55 |              -0.73 | Por debajo del promedio |               0.00
 Jake Westbrook     | Platinum         |              16.29 |           3.58 |              12.71 | Por encima del promedio |               1.00
 Jensen Lewis       | Platinum         |              15.25 |           3.58 |              11.67 | Por encima del promedio |               0.92
 David Dellucci     | Platinum         |              12.38 |           3.58 |               8.80 | Por encima del promedio |               0.83
 Kelly Shoppach     | Platinum         |              11.88 |           3.58 |               8.30 | Por encima del promedio |               0.75
 Cliff Lee          | Platinum         |               7.75 |           3.58 |               4.17 | Por encima del promedio |               0.67
 Victor Martinez    | Platinum         |               3.76 |           3.58 |               0.18 | Por encima del promedio |               0.58
 ...
(122 rows; se muestran las primeras 20)
```

In [12]:
# Detener Spark
spark.stop()
print("\nETL DataMart PlayerPerformance COMPLETADO")


ETL DataMart PlayerPerformance COMPLETADO
